In [1]:
import os
import json
import csv
import math
import random

from PIL import Image, ImageDraw

IMG_W = 512
IMG_H = 512
MARGIN = 20

SHAPES = ["square", "circle", "triangle"]
COLORS = {
    "red":    (220, 52, 44),
    "green":  (58, 164, 74),
    "blue":   (47, 95, 208),
    "yellow": (224, 176, 32),
    "cyan":   (43, 184, 196),
}
SIZES = {"small": 26, "big": 46}

MIN_OBJECTS = 3
MAX_OBJECTS = 10

GRID_STEP = 32
GRID_COLOR = (232, 234, 238)


def shape_metrics(shape, r):
    if shape == "circle":
        area = math.pi * r * r
        dimension = {"name": "radius", "value": round(r, 1)}
    elif shape == "square":
        side = 2 * r
        area = side * side
        dimension = {"name": "side", "value": round(side, 1)}
    else:
        side = r * math.sqrt(3)
        area = (math.sqrt(3) / 4) * side * side
        dimension = {"name": "side", "value": round(side, 1)}
    return round(area, 1), dimension


def overlaps(cx, cy, r, placed):
    for (px, py, pr) in placed:
        if math.hypot(cx - px, cy - py) < (r + pr + 10):
            return True
    return False


def draw_grid(draw):
    for x in range(0, IMG_W + 1, GRID_STEP):
        draw.line([(x, 0), (x, IMG_H)], fill=GRID_COLOR, width=1)
    for y in range(0, IMG_H + 1, GRID_STEP):
        draw.line([(0, y), (IMG_W, y)], fill=GRID_COLOR, width=1)


def draw_shape(draw, shape, cx, cy, r, fill):
    if shape == "circle":
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fill, outline=(30, 30, 30), width=2)
    elif shape == "square":
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=fill, outline=(30, 30, 30), width=2)
    else:
        pts = []
        for k in range(3):
            ang = -math.pi / 2 + k * (2 * math.pi / 3)
            pts.append((cx + r * math.cos(ang), cy + r * math.sin(ang)))
        draw.polygon(pts, fill=fill, outline=(30, 30, 30))


def generate_one():
    img = Image.new("RGB", (IMG_W, IMG_H), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    draw_grid(draw)

    n = random.randint(MIN_OBJECTS, MAX_OBJECTS)
    placed = []
    objects = []

    attempts = 0
    while len(objects) < n and attempts < 1200:
        attempts += 1
        shape = random.choice(SHAPES)
        color = random.choice(list(COLORS.keys()))
        size = random.choice(list(SIZES.keys()))
        r = SIZES[size]

        cx = random.randint(MARGIN + r, IMG_W - MARGIN - r)
        cy = random.randint(MARGIN + r, IMG_H - MARGIN - r)

        if overlaps(cx, cy, r, placed):
            continue

        placed.append((cx, cy, r))
        draw_shape(draw, shape, cx, cy, r, COLORS[color])

        area, dimension = shape_metrics(shape, r)
        objects.append({
            "shape": shape,
            "color": color,
            "size": size,
            "center": [cx, cy],
            "area": area,
            "dimension": dimension,
        })

    return img, objects


def build_dataset(out_dir, num_images):
    image_dir = os.path.join(out_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    records = []
    csv_rows = []

    for i in range(num_images):
        img, objects = generate_one()
        fn = f"shapes_{i:05d}.png"
        img.save(os.path.join(image_dir, fn))

        records.append({
            "image_filename": fn,
            "num_objects": len(objects),
            "objects": objects,
        })

        for obj in objects:
            csv_rows.append({
                "image_filename": fn,
                "shape": obj["shape"],
                "color": obj["color"],
                "size": obj["size"],
                "center_x": obj["center"][0],
                "center_y": obj["center"][1],
                "area": obj["area"],
                "dimension_name": obj["dimension"]["name"],
                "dimension_value": obj["dimension"]["value"],
            })

        if i % 200 == 0:
            print(f"generated {i} images...")

    with open(os.path.join(out_dir, "annotations.json"), "w") as f:
        json.dump(records, f, indent=2)

    with open(os.path.join(out_dir, "annotations.csv"), "w", newline="") as f:
        fieldnames = ["image_filename", "shape", "color", "size",
                      "center_x", "center_y", "area",
                      "dimension_name", "dimension_value"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)

    print("done.")
    print("images:", num_images)
    print("total objects:", len(csv_rows))

    return records


# build_dataset("shapes_data", num_images=4000);

In [2]:
import random

SHAPES = ["square", "circle", "triangle"]
COLORS = ["red", "green", "blue", "yellow", "cyan"]
SIZES = ["small", "big"]


def unique_referent(objects):
    pool = list(objects)
    random.shuffle(pool)
    for o in pool:
        m = [x for x in objects if x["color"] == o["color"] and x["shape"] == o["shape"]]
        if len(m) == 1:
            return f"the {o['color']} {o['shape']}", o
    for o in pool:
        m = [x for x in objects if x["color"] == o["color"] and x["shape"] == o["shape"] and x["size"] == o["size"]]
        if len(m) == 1:
            return f"the {o['size']} {o['color']} {o['shape']}", o
    return None, None


def q_count_total(objs):
    return "How many objects are there?", str(len(objs))

def q_count_color(objs):
    c = random.choice(COLORS)
    return f"How many {c} objects are there?", str(sum(o["color"] == c for o in objs))

def q_count_shape(objs):
    s = random.choice(SHAPES)
    return f"How many {s}s are there?", str(sum(o["shape"] == s for o in objs))

def q_count_size(objs):
    z = random.choice(SIZES)
    return f"How many {z} objects are there?", str(sum(o["size"] == z for o in objs))

def q_count_color_shape(objs):
    c, s = random.choice(COLORS), random.choice(SHAPES)
    return f"How many {c} {s}s are there?", str(sum(o["color"] == c and o["shape"] == s for o in objs))

def q_exist(objs):
    c, s = random.choice(COLORS), random.choice(SHAPES)
    return f"Is there a {c} {s}?", "Yes" if any(o["color"] == c and o["shape"] == s for o in objs) else "No"

# def q_compare_counts(objs):
#     s1, s2 = random.sample(SHAPES, 2)
#     n1 = sum(o["shape"] == s1 for o in objs)
#     n2 = sum(o["shape"] == s2 for o in objs)
#     if n1 == n2:
#         return f"Are there an equal number of {s1}s and {s2}s?", "Yes"
#     return f"Are there more {s1}s than {s2}s?", "Yes" if n1 > n2 else "No"

# def q_most_common_shape(objs):
#     counts = {s: sum(o["shape"] == s for o in objs) for s in SHAPES}
#     best = max(counts, key=counts.get)
#     top = [s for s, n in counts.items() if n == counts[best]]
#     if len(top) != 1:
#         return None
#     return "Which shape appears most often?", best

# def q_attr_of_referent(objs):
#     phrase, ref = unique_referent(objs)
#     if ref is None:
#         return None
#     attr = random.choice(["color", "size", "shape"])
#     return f"What is the {attr} of {phrase}?", ref[attr]

def q_area_of_referent(objs):
    phrase, ref = unique_referent(objs)
    if ref is None:
        return None
    return f"What is the area of {phrase}?", str(ref["area"])

def q_dimension_of_referent(objs):
    phrase, ref = unique_referent(objs)
    if ref is None:
        return None
    d = ref["dimension"]
    return f"What is the {d['name']} of {phrase}?", str(d["value"])

def q_relational(objs):
    if len(objs) < 2:
        return None
    phrase, ref = unique_referent(objs)
    if ref is None:
        return None
    rel = random.choice(["right of", "left of", "above", "below"])
    rx, ry = ref["center"]
    def passes(o):
        if o is ref: return False
        ox, oy = o["center"]
        return {"right of": ox > rx, "left of": ox < rx, "above": oy < ry, "below": oy > ry}[rel]
    mode = random.choice(["all", "color", "shape"])
    if mode == "color":
        c = random.choice(COLORS)
        return f"How many {c} objects are {rel} {phrase}?", str(sum(passes(o) and o["color"] == c for o in objs))
    if mode == "shape":
        s = random.choice(SHAPES)
        return f"How many {s}s are {rel} {phrase}?", str(sum(passes(o) and o["shape"] == s for o in objs))
    return f"How many objects are {rel} {phrase}?", str(sum(passes(o) for o in objs))

def q_yesno_relational(objs):
    if len(objs) < 2:
        return None
    phrase, ref = unique_referent(objs)
    if ref is None:
        return None
    rel = random.choice(["right of", "left of", "above", "below"])
    rx, ry = ref["center"]
    c, s = random.choice(COLORS), random.choice(SHAPES)
    def passes(o):
        if o is ref: return False
        ox, oy = o["center"]
        base = {"right of": ox > rx, "left of": ox < rx, "above": oy < ry, "below": oy > ry}[rel]
        return base and o["color"] == c and o["shape"] == s
    return f"Is there a {c} {s} {rel} {phrase}?", "Yes" if any(passes(o) for o in objs) else "No"

# def q_largest(objs):
#     mx = max(o["area"] for o in objs)
#     top = [o for o in objs if o["area"] == mx]
#     if len(top) != 1:
#         return None
#     o = top[0]
#     return "What is the largest object?", f"{o['color']} {o['shape']}"

# def q_smallest(objs):
#     mn = min(o["area"] for o in objs)
#     top = [o for o in objs if o["area"] == mn]
#     if len(top) != 1:
#         return None
#     o = top[0]
#     return "What is the smallest object?", f"{o['color']} {o['shape']}"

# def q_closest_topleft(objs):
#     key = lambda o: (o["center"][0] ** 2 + o["center"][1] ** 2)
#     mn = min(key(o) for o in objs)
#     top = [o for o in objs if key(o) == mn]
#     if len(top) != 1:
#         return None
#     o = top[0]
#     return "Which object is closest to the top-left corner?", f"{o['color']} {o['shape']}"


# CATEGORY = [q_count_total, q_count_color, q_count_shape, q_count_size,
#             q_count_color_shape, q_exist, q_compare_counts, q_most_common_shape]
# REFERENT = [q_attr_of_referent, q_area_of_referent, q_dimension_of_referent,
#             q_relational, q_yesno_relational]
# SUPER = [q_largest, q_smallest, q_closest_topleft]
# ALL = CATEGORY + REFERENT + SUPER

CATEGORY = [q_count_total, q_count_color, q_count_shape, q_count_size,
            q_count_color_shape, q_exist]
REFERENT = [ q_area_of_referent, q_dimension_of_referent,
            q_relational, q_yesno_relational]
ALL = CATEGORY + REFERENT 


def make_questions(objects, n=10):
    qa, seen, tries = [], set(), 0
    while len(qa) < n and tries < n * 12:
        tries += 1
        gen = random.choice(ALL)
        res = gen(list(objects))
        if res is None:
            continue
        q, a = res
        if q in seen:
            continue
        seen.add(q)
        qa.append({"question": q, "answer": str(a)})
    return qa


import json
import random
# from generate_qa import make_questions

def build_training(annotations_path, out_path, questions_per_image=10):
    random.seed(0)
    with open(annotations_path) as f:
        records = json.load(f)

    examples = []
    for rec in records:
        objects = rec["objects"]
        for qa in make_questions(objects, n=questions_per_image):
            target = json.dumps({"answer": qa["answer"], "objects": objects})
            examples.append({
                "image_filename": rec["image_filename"],
                "question": qa["question"],
                "target": target
            })

    with open(out_path, "w") as f:
        json.dump(examples, f, indent=2)

    print("images:", len(records))
    print("training examples:", len(examples))
    return examples

# if __name__ == "__main__":
# build_training("shapes_data/annotations.json", "shapes_data/train_examples.json");


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import random
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model


TRAIN_FILE = "shapes_data/train_examples.json"
IMAGE_DIR = "shapes_data/images"
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_OUT = "shapes_lora_adapter"

EVAL_EVERY = 400         # evaluate on val set this often
VAL_SIZE = 500           # held-out examples


model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.image_processor.max_pixels = 256 * 28 * 28
model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


class ShapesDataset(Dataset):
    def __init__(self, examples, image_dir):
        self.examples = examples
        self.image_dir = image_dir
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        ex = self.examples[i]
        image = Image.open(f"{self.image_dir}/{ex['image_filename']}").convert("RGB")
        return {"image": image, "question": ex["question"], "target": ex["target"]}


def collate_fn(batch):
    images, prompt_texts, full_texts = [], [], []
    for ex in batch:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": ex["image"]},
            {"type": "text", "text": ex["question"]}
        ]}]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images.append(ex["image"])
        prompt_texts.append(prompt)
        full_texts.append(prompt + ex["target"])
    model_inputs = processor(text=full_texts, images=images, padding=True, return_tensors="pt")
    labels = model_inputs["input_ids"].clone()
    for i in range(len(batch)):
        prompt_only = processor(text=[prompt_texts[i]], images=[images[i]], return_tensors="pt")
        prompt_len = prompt_only["input_ids"].shape[1]
        labels[i, :prompt_len] = -100
    labels[labels == processor.tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels
    return model_inputs


# split train / val
with open(TRAIN_FILE) as f:
    all_examples = json.load(f)
random.seed(0)
random.shuffle(all_examples)
val_examples = all_examples[:VAL_SIZE]
train_examples = all_examples[VAL_SIZE:]
print("train:", len(train_examples), " val:", len(val_examples))

train_dataset = ShapesDataset(train_examples, IMAGE_DIR)
val_dataset = ShapesDataset(val_examples, IMAGE_DIR)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


@torch.no_grad()
def evaluate():
    model.eval()
    total, n = 0.0, 0
    for batch in val_loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss
        total += loss.item()
        n += 1
    model.train()
    return total / max(n, 1)


NUM_EPOCHS = 1
best_val = float("inf")

model.train()
for epoch in range(NUM_EPOCHS):
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"epoch {epoch}  step {step}  train_loss {loss.item():.4f}")

        if step % EVAL_EVERY == 0 and step > 0:
            val_loss = evaluate()
            flag = ""
            if val_loss < best_val:
                best_val = val_loss
                model.save_pretrained(ADAPTER_OUT)
                flag = "  <- saved (new best)"
            print(f"           val_loss {val_loss:.4f}  (best {best_val:.4f}){flag}")

# final eval + save if best
val_loss = evaluate()
if val_loss < best_val:
    best_val = val_loss
    model.save_pretrained(ADAPTER_OUT)
    print(f"final val_loss {val_loss:.4f}  <- saved (new best)")
print("training done. best val_loss:", round(best_val, 4))

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,758,309,376 || trainable%: 0.0981
train: 39500  val: 500


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


epoch 0  step 0  train_loss 0.8664
epoch 0  step 50  train_loss 0.2727
epoch 0  step 100  train_loss 0.1961
epoch 0  step 150  train_loss 0.1486
epoch 0  step 200  train_loss 0.1623
epoch 0  step 250  train_loss 0.1285
epoch 0  step 300  train_loss 0.1297
epoch 0  step 350  train_loss 0.1441
epoch 0  step 400  train_loss 0.1249
           val_loss 0.1310  (best 0.1310)  <- saved (new best)
epoch 0  step 450  train_loss 0.1167
epoch 0  step 500  train_loss 0.1095
epoch 0  step 550  train_loss 0.1315
epoch 0  step 600  train_loss 0.1213
epoch 0  step 650  train_loss 0.1406
epoch 0  step 700  train_loss 0.1084
epoch 0  step 750  train_loss 0.1301
epoch 0  step 800  train_loss 0.1327
           val_loss 0.1202  (best 0.1202)  <- saved (new best)
epoch 0  step 850  train_loss 0.1409
epoch 0  step 900  train_loss 0.1228
epoch 0  step 950  train_loss 0.1210
epoch 0  step 1000  train_loss 0.1164
epoch 0  step 1050  train_loss 0.1208
epoch 0  step 1100  train_loss 0.1003
epoch 0  step 1150  tra

KeyboardInterrupt: 